# Brain-to-Text Metrics v2

This notebook evaluates NeuroVLM brain-to-text generation separately from text-to-brain. It keeps the semantic metrics from `new_neuropvlm_metrics_v2`, removes the cleaned abstract branch, adds Recall@1 retrieval, and adds network label accuracy for the Networks test set.

This notebook is configured to run the full available networks, PubMed test, and NeuroVault evaluation sets by default. For fast iteration, set `MAX_B2T` to a small integer.

In [ ]:
import os
import hashlib
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from neurovlm import NeuroVLM
from neurovlm.data import load_dataset, load_latent
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoModel, AutoTokenizer


In [ ]:
LLM_BACKEND = "huggingface"
LLM_MODEL = "qwen2.5:7b-instruct"
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"
MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MPNET_MODEL = "sentence-transformers/all-mpnet-base-v2"
B2T_RANDOM_EVIDENCE_N_SAMPLES = 25
B2T_RANDOM_EVIDENCE_SEED = 13

MAX_B2T = None        # full available networks, PubMed test, and NeuroVault sets
RUN_NETWORKS = True
RUN_PUBMED = True
RUN_NEUROVAULT = True
RUN_GENERATED_TEXT = False  # slow LLM/BERTScore generated-text evaluation

B2T_TOP_K = 5
B2T_SIM_THR = 0.35
B2T_DATASETS = ["llm_neuro_terms", "kg_mesh", "cogatlas"]
from neurovlm.evaluation_notebook_utils import resolve_evaluation_output_dir

OUTPUT_DIR = resolve_evaluation_output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
nvlm = NeuroVLM()
B2T_ENCODER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_st_model = SentenceTransformer(MINILM_MODEL, device=B2T_ENCODER_DEVICE)
_mpnet_model = SentenceTransformer(MPNET_MODEL, device=B2T_ENCODER_DEVICE)
print(f"Ready. Sentence encoders on {B2T_ENCODER_DEVICE}.")


In [ ]:
from neurovlm.evaluation_notebook_utils import load_network_label_resources

network_labels_df, network_info, NETWORK_TEST_SET_SOURCE = load_network_label_resources()

DISPLAY_TO_KEY = dict(zip(network_info["display"], network_info["network_key"]))
KEY_TO_DISPLAY = dict(zip(network_info["network_key"], network_info["display"]))
SHORT_LABELS = dict(zip(network_info["display"], network_info["short_definition"]))
LONG_LABELS = dict(zip(network_info["display"], network_info["long_definition"]))

print(f"Loaded {len(network_info)} canonical network labels from {NETWORK_TEST_SET_SOURCE}")
display(network_info[["network_key", "display", "short_definition"]])


In [ ]:
from neurovlm.evaluation_notebook_utils import build_labeled_network_data

networks_data, all_net_latents = build_labeled_network_data(network_labels_df)

print(f"Networks loaded: {len(networks_data)} labeled maps across {len(all_net_latents)} atlases")
display(pd.DataFrame([
    {"sample": name, "network_key": d["network_key"], "display": d["display"]}
    for name, d in networks_data.items()
]).head())


In [ ]:
from neurovlm.evaluation_notebook_utils import build_pubmed_b2t_eval

pubmed_eval, pubmed_stats = build_pubmed_b2t_eval(MAX_B2T)
if pubmed_stats["missing_summary_latents"]:
    print(f"Skipped {pubmed_stats['missing_summary_latents']:,} PubMed test papers without precomputed summary text latents.")
print(f"PubMed summary test samples: {pubmed_stats['eval_records']} / {pubmed_stats['records']}")
print(f"Summaries table rows: {pubmed_stats['summary_rows']:,}; test rows: {pubmed_stats['test_rows']:,}")


In [ ]:
from neurovlm.evaluation_notebook_utils import build_neurovault_b2t_eval

neurovault_eval, neurovault_selection_df = build_neurovault_b2t_eval(nvlm, OUTPUT_DIR, MAX_B2T)
print(f"NeuroVault samples: {len(neurovault_eval)} / {len(neurovault_selection_df)}")
print("NeuroVault image selection: argmax brain-image/shared-text similarity per DOI.")


## Metric Helpers

`nvlm_sim` is a cosine similarity in NeuroVLM's shared latent space. A value around 0.33 can be meaningful because it is measured against a broad retrieval space, so the notebook plots it with empirical quartiles and a random-pair baseline rather than treating 1.0 as the only intuitive target.

In [ ]:
from neurovlm.brain_to_text_metrics import make_b2t_runner

SHORT_PROMPT_GENERAL = (
    "Reply with a single concise sentence (10-20 words) naming the main cognitive "
    "function or brain network. Output only that sentence."
)
SHORT_PROMPT_PUBMED = (
    "Generate only a paper title (6-12 words) for the neuroimaging study this "
    "brain activation pattern represents. Output the title only."
)
LONG_PROMPT = ""

run_b2t = make_b2t_runner(
    nvlm=nvlm,
    st_model=_st_model,
    st_util=st_util,
    bert_score_fn=bert_score,
    bertscore_model=BERTSCORE_MODEL,
    llm_backend=LLM_BACKEND,
    llm_model=LLM_MODEL,
    b2t_datasets=B2T_DATASETS,
    b2t_top_k=B2T_TOP_K,
    b2t_sim_threshold=B2T_SIM_THR,
)


In [ ]:
from neurovlm.brain_to_text_metrics import make_network_label_accuracy_adder

add_network_label_accuracy = make_network_label_accuracy_adder(
    networks_data=networks_data,
    network_info=network_info,
    st_model=_st_model,
    st_util=st_util,
)


## Retrieval Evidence Metrics

These analyses evaluate retrieval before any LLM generation. That keeps the metric tied to what the brain encoder ranks, rather than to how well the LLM writes from those ranked results.

- **Paper-level retrieval** is the canonical PubMed metric and is also applied to NeuroVault. It asks whether each brain embedding retrieves its paired paper text, with exact-paper and semantic-neighbor variants.
- **Network gold-term ranking** asks whether known network terms appear high in the ranked NeuroVLM term list. The notebook keeps the broad `llm_neuro_terms + kg_mesh + cogatlas` corpus; normalized k keeps this comparable to a network-CSV-only term corpus.

Generated text evaluation is kept later and is disabled by default because it is much slower.


### PubMed Retrieval Corpus

PubMed paper-level retrieval is the primary PubMed metric. This setting only controls the MeSH corpus used later if generated-text context is retrieved for PubMed examples.


In [ ]:
INCLUDE_MOLECULAR_MESH = False
MESH_BRAIN_RANKABLE_NODE_TYPES = [
    "disorder",
    "anatomical_region",
    "biological_process",
    "cognitive_construct",
]
if INCLUDE_MOLECULAR_MESH:
    MESH_BRAIN_RANKABLE_NODE_TYPES.append("molecular")

PUBMED_B2T_DATASET = "kg_mesh_brain_rankable_plus_molecular" if INCLUDE_MOLECULAR_MESH else "kg_mesh_brain_rankable"
print(f"PubMed generated-text retrieval corpus: {PUBMED_B2T_DATASET}")
print(f"Allowed PubMed MeSH node types in that corpus: {MESH_BRAIN_RANKABLE_NODE_TYPES}")


In [ ]:
from neurovlm.brain_to_text_metrics import (
    dataset_records_for_retrieval_eval,
    full_retrieval_table_for_sample,
    network_gold_terms,
    normalize_term_text,
    pubmed_abstract_lookup,
    terms_for_dataset,
    unique_ranked_terms_from_table,
)

TERM_EVAL_NORMALIZED_KS = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10]
TERM_RECALL_CURVE_NORMALIZED_KS = np.linspace(0.0, 1.0, 101)
B2T_TERM_EXAMPLE_TOP_K = 20
B2T_TERM_TOP_K = B2T_TERM_EXAMPLE_TOP_K
B2T_EVIDENCE_TOP_K = B2T_TOP_K
NETWORK_B2T_TERM_DATASETS = ["llm_neuro_terms", "kg_mesh", "cogatlas"]
B2T_TERM_DATASETS_BY_EVAL_DATASET = {
    "networks": NETWORK_B2T_TERM_DATASETS,
    "pubmed": [PUBMED_B2T_DATASET],
    "neurovault": NETWORK_B2T_TERM_DATASETS,
}
B2T_RETRIEVAL_TABLE_CACHE = {}
PUBMED_ABSTRACT_LOOKUP = pubmed_abstract_lookup()


### Paper-Level Retrieval: PubMed and NeuroVault

This is the canonical retrieval metric for PubMed papers and is also applied to NeuroVault. It asks whether each brain embedding retrieves its paired paper text from the held-out candidate set, before any LLM generation.

The paper text embeddings come from the precomputed Hugging Face latent files (`pubmed_summaries` and `neurovault_text`), then use the NeuroVLM text projection head. They are not re-encoded in the notebook. For NeuroVault, the representative brain map for each DOI is selected by argmax similarity between candidate image embeddings and the paired publication text embedding.

Metric columns:

- `brain_to_paper_normalized_k_recall_curve_auc`: strict exact-pair retrieval from brain embeddings to paper text embeddings, integrated over `k / n_papers`.
- `paper_to_brain_normalized_k_recall_curve_auc`: reverse exact-pair retrieval from paper text embeddings to brain embeddings.
- `semantic_normalized_k_recall_curve_auc`: paper retrieval where the exact paper and its nearest text neighbors count as positives; this is less brittle than exact PMID/DOI matching.


In [ ]:
from neurovlm.brain_to_text_metrics import run_paper_retrieval_evaluations

PAPER_RETRIEVAL_BRAIN_BATCH_SIZE = 512
PAPER_RETRIEVAL_TEXT_BATCH_SIZE = 512
PAPER_RETRIEVAL_SEMANTIC_NEIGHBORS = 10

b2t_paper_retrieval_metrics_df, b2t_paper_retrieval_curves_df, b2t_paper_retrieval_examples_df = run_paper_retrieval_evaluations(
    nvlm=nvlm,
    pubmed_eval=pubmed_eval,
    neurovault_eval=neurovault_eval,
    output_dir=OUTPUT_DIR,
    run_pubmed=RUN_PUBMED,
    run_neurovault=RUN_NEUROVAULT,
    semantic_neighbors=PAPER_RETRIEVAL_SEMANTIC_NEIGHBORS,
    brain_batch_size=PAPER_RETRIEVAL_BRAIN_BATCH_SIZE,
    text_batch_size=PAPER_RETRIEVAL_TEXT_BATCH_SIZE,
)

if len(b2t_paper_retrieval_metrics_df):
    display(b2t_paper_retrieval_metrics_df.round(3))
else:
    print("No PubMed/NeuroVault paper retrieval metrics were computed.")


### Paper Retrieval Visualization

Each dataset gets its own plot with three recall curves: strict brain-to-paper retrieval, strict paper-to-brain retrieval, and semantic-neighbor brain-to-paper retrieval. The legend reports normalized-k recall-curve AUC for each curve. Curves above the dashed diagonal retrieve relevant papers earlier than random ranking.


In [ ]:
if len(b2t_paper_retrieval_curves_df):
    metric_labels = [
        ("brain_to_paper_recall_curve", "brain -> paper", "brain_to_paper_normalized_k_recall_curve_auc"),
        ("paper_to_brain_recall_curve", "paper -> brain", "paper_to_brain_normalized_k_recall_curve_auc"),
        ("semantic_recall_curve", "semantic neighbors", "semantic_normalized_k_recall_curve_auc"),
    ]
    for dataset, sub in b2t_paper_retrieval_curves_df.groupby("dataset"):
        metrics_row = b2t_paper_retrieval_metrics_df.set_index("dataset").loc[dataset]
        fig, ax = plt.subplots(figsize=(7, 5))
        for curve_col, label, auc_col in metric_labels:
            auc = float(metrics_row[auc_col])
            ax.plot(sub["normalized_k"], sub[curve_col], label=f"{label} (AUC={auc:.3f})")
        ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random chance")
        ax.set_xlabel("Normalized k (k / n papers)")
        ax.set_ylabel("Recall@k")
        ax.set_title(f"{dataset}: Paper-Level Brain/Text Retrieval")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
        fig.tight_layout()
        plt.savefig(OUTPUT_DIR / f"b2t_paper_retrieval_curves_{dataset}.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No paper retrieval curves to plot.")


### Network Gold-Term Ranking

This evaluates whether the full ranked NeuroVLM term list retrieves known network terms from the network test-set CSV. The candidate corpus is intentionally broad (`llm_neuro_terms`, `kg_mesh`, and `cogatlas`), while the gold terms come from the CSV. A CSV-term-only candidate corpus should lead to the same interpretation when results are reported against normalized k; this broader corpus is the stricter retrieval setting.

Metric columns:

- `normalized_k_target`: the requested fraction of the candidate corpus to inspect, such as `0.01` for the top 1%.
- `k`: the actual number of retrieved terms inspected, computed as `ceil(normalized_k_target * n_candidate_terms)`.
- `n_candidate_terms`: the number of unique terms the model could rank after normalizing/deduplicating term names.
- `n_gold_terms`: gold terms that are reachable in the candidate corpus. Unreachable terms are excluded so the model is not penalized for terms it could never retrieve.
- `n_unreachable_gold_terms`: gold terms missing from the retrieval corpora; this is a fairness/audit check.
- `precision_at_normalized_k`: among the inspected top-k terms, the fraction that are gold terms.
- `recall_at_normalized_k`: among the reachable gold terms, the fraction recovered within the inspected top-k terms.
- `hit_at_normalized_k`: whether at least one gold term appears in the inspected top-k terms.
- `mrr_at_normalized_k`: reciprocal rank of the first gold-term hit if it appears within top-k.
- `normalized_first_hit_rank`: first gold-term rank divided by `n_candidate_terms`; lower is better and accounts for corpus size.
- `expected_random_recall_at_normalized_k`: chance-level recall if terms were randomly ordered.
- `recall_auc`: area under the recall-vs-normalized-k curve. Higher means gold terms appear earlier across the full ranked list.

The recall curve plots `recall_at_normalized_k` on the y-axis and normalized k on the x-axis. A curve above the diagonal means the brain-to-text retrieval ranks gold terms earlier than random chance.


In [ ]:
from neurovlm.brain_to_text_metrics import exact_term_ranking_outputs

network_candidate_terms = set().union(*(terms_for_dataset(ds) for ds in NETWORK_B2T_TERM_DATASETS))
network_missing_rows = []
for sample_name in networks_data:
    for term in network_gold_terms(sample_name, networks_data, network_labels_df):
        norm = normalize_term_text(term)
        if norm and norm not in network_candidate_terms:
            network_missing_rows.append({"sample": sample_name, "term": term, "normalized_term": norm})
network_missing_terms_df = pd.DataFrame(network_missing_rows).drop_duplicates() if network_missing_rows else pd.DataFrame(columns=["sample", "term", "normalized_term"])
display(network_missing_terms_df.head(25))
print(f"Network gold terms missing from llm_neuro_terms + kg_mesh + cogatlas: {len(network_missing_terms_df):,}")
if len(network_missing_terms_df):
    network_missing_terms_df.to_csv(OUTPUT_DIR / "network_gold_terms_missing_from_retrieval_corpora.csv", index=False)

pubmed_candidate_terms = terms_for_dataset(PUBMED_B2T_DATASET)


In [ ]:
from neurovlm.brain_to_text_metrics import run_network_gold_term_ranking

b2t_term_metrics_df, b2t_term_recall_curve_df, b2t_term_auc_df, b2t_term_examples_df = run_network_gold_term_ranking(
    nvlm=nvlm,
    networks_data=networks_data,
    network_labels_df=network_labels_df,
    pubmed_eval=pubmed_eval,
    neurovault_eval=neurovault_eval,
    pubmed_abs_lookup=PUBMED_ABSTRACT_LOOKUP,
    network_candidate_terms=network_candidate_terms,
    term_datasets_by_eval_dataset=B2T_TERM_DATASETS_BY_EVAL_DATASET,
    retrieval_table_cache=B2T_RETRIEVAL_TABLE_CACHE,
    term_eval_normalized_ks=TERM_EVAL_NORMALIZED_KS,
    term_recall_curve_normalized_ks=TERM_RECALL_CURVE_NORMALIZED_KS,
    b2t_term_example_top_k=B2T_TERM_EXAMPLE_TOP_K,
    output_dir=OUTPUT_DIR,
    run_networks=RUN_NETWORKS,
    run_pubmed=RUN_PUBMED,
    run_neurovault=RUN_NEUROVAULT,
)

if len(b2t_term_metrics_df):
    display(
        b2t_term_metrics_df
        .groupby(["dataset", "normalized_k_target"])[
            [
                "normalized_k", "k", "n_candidate_terms",
                "precision_at_normalized_k", "recall_at_normalized_k",
                "hit_at_normalized_k", "mrr_at_normalized_k",
                "expected_random_recall_at_normalized_k", "n_unreachable_gold_terms",
            ]
        ]
        .mean()
        .round(3)
    )
    display(b2t_term_auc_df.groupby("dataset")[["recall_auc", "expected_random_recall_auc", "recall_auc_minus_random"]].agg(["mean", "std", "count"]).round(3))
else:
    print("No network gold-term metrics were computed.")


### Network Term-Ranking Visualization

This plot shows how quickly known network gold terms appear as the normalized term budget grows. Because the x-axis is `k / n_candidate_terms`, the curve remains interpretable even when comparing the broad NeuroVLM corpus with a smaller CSV-term-only corpus.


In [ ]:
if len(b2t_term_recall_curve_df):
    fig, ax = plt.subplots(figsize=(7, 5))
    curve_summary = (
        b2t_term_recall_curve_df
        .groupby(["dataset", "normalized_k_target"])
        .agg(recall_mean=("recall_at_normalized_k", "mean"), recall_std=("recall_at_normalized_k", "std"), n=("sample", "count"))
        .reset_index()
    )
    auc_means = b2t_term_auc_df.groupby("dataset")["recall_auc"].mean().to_dict()
    for dataset, sub in curve_summary.groupby("dataset"):
        label = f"{dataset} (AUC={auc_means.get(dataset, np.nan):.2f})"
        ax.plot(sub["normalized_k_target"], sub["recall_mean"], label=label)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random chance")
    ax.set_xlabel("Normalized k (k / n candidate terms)")
    ax.set_ylabel("Recall@k")
    ax.set_title("Network Normalized Gold-Term Recall Curve")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_approach1_normalized_recall_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No recall curve rows to plot.")


In [ ]:
from neurovlm.brain_to_text_metrics import predownload_hf_model

if RUN_GENERATED_TEXT:
    predownload_hf_model(BERTSCORE_MODEL, AutoTokenizer, AutoModel)
else:
    print("Skipping BERTScore model pre-download because RUN_GENERATED_TEXT = False.")


### PubMed MeSH Gold-Term Ranking

This is a term-recovery diagnostic for PubMed, separate from the primary paper-level retrieval metric. It uses the same multi-positive ranking definition as `evaluate_mesh_term_ranking`: for each paper, any allowed gold MeSH term can be the first positive hit, and the recall-curve AUC is computed from the best true-term rank over the full candidate list.

Both the candidate corpus and the ground-truth MeSH terms are restricted to the brain-rankable MeSH node types: `disorder`, `anatomical_region`, `biological_process`, and `cognitive_construct` by default. This should line up with `mesh_paper_recall_curve_auc` / `mesh_normalized_k_recall_curve_auc` from the semantic baseline run.


In [ ]:
from neurovlm.brain_to_text_metrics import run_pubmed_mesh_gold_ranking

b2t_mesh_term_metrics_df, b2t_mesh_term_recall_curve_df, b2t_mesh_term_examples_df = run_pubmed_mesh_gold_ranking(
    nvlm=nvlm,
    pubmed_eval=pubmed_eval,
    pubmed_b2t_dataset=PUBMED_B2T_DATASET,
    mesh_brain_rankable_node_types=MESH_BRAIN_RANKABLE_NODE_TYPES,
    b2t_term_example_top_k=B2T_TERM_EXAMPLE_TOP_K,
    output_dir=OUTPUT_DIR,
    run_pubmed=RUN_PUBMED,
)

if len(b2t_mesh_term_metrics_df):
    display(b2t_mesh_term_metrics_df.round(3))
else:
    print("No PubMed MeSH gold-term metrics were computed.")


### PubMed MeSH Gold-Term Visualization

This plot shows MeSH gold-term recall as the normalized MeSH candidate budget grows. It is useful as a term-level diagnostic, while the paper-level retrieval plots remain the primary PubMed paper evaluation.


In [ ]:
if len(b2t_mesh_term_recall_curve_df):
    fig, ax = plt.subplots(figsize=(7, 5))
    auc = float(b2t_mesh_term_metrics_df.loc[0, "mesh_normalized_k_recall_curve_auc"])
    ax.plot(
        b2t_mesh_term_recall_curve_df["normalized_k"],
        b2t_mesh_term_recall_curve_df["recall_at_normalized_k"],
        label=f"pubmed_mesh (AUC={auc:.3f})",
    )
    ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random chance")
    ax.set_xlabel("Normalized k (k / n MeSH candidate terms)")
    ax.set_ylabel("Best-positive MeSH recall@k")
    ax.set_title("PubMed MeSH Normalized Gold-Term Recall Curve")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_pubmed_mesh_normalized_recall_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No PubMed MeSH recall curve rows to plot.")


## Generated Text Evaluation

This optional section compares LLM-generated text against the ground-truth text. It is disabled by default with `RUN_GENERATED_TEXT = False` because it is much slower than the retrieval metrics.

Metrics in this section, when enabled:

- `bert_p`, `bert_r`, `bert_f1`: BERTScore precision, recall, and F1 between generated text and ground truth. These check token-level contextual semantic overlap.
- `sem_sim`: MiniLM sentence-embedding similarity between generated text and ground truth.
- `nvlm_sim`: cosine similarity between the generated text embedding and the brain query in NeuroVLM's shared latent space.
- `generated_text_normalized_k_recall_curve_auc`: embeds each generated text and asks how early it retrieves its source brain map within the dataset/mode group, summarized as area under Recall@k over normalized `k / n`.
- Network label accuracy metrics: for Networks, these check whether the generated text names the correct network label by alias matching or semantic matching.

Because generation depends on the LLM as well as the retrieval results, this section should be interpreted alongside the paper-level retrieval and network term-ranking metrics.


In [ ]:
if RUN_GENERATED_TEXT:
    b2t_frames = []

    if RUN_NETWORKS:
        records = []
        for net_name, d in tqdm(networks_data.items(), desc="Networks B2T"):
            records.extend(run_b2t(net_name, d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_GENERAL, LONG_PROMPT))
        b2t_net_df = add_network_label_accuracy(pd.DataFrame(records))
        b2t_net_df["dataset"] = "networks"
        b2t_frames.append(b2t_net_df)

    if RUN_PUBMED:
        records = []
        for d in tqdm(pubmed_eval, desc="PubMed B2T"):
            records.extend(run_b2t(str(d["pmid"]), d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_PUBMED, LONG_PROMPT, datasets=[PUBMED_B2T_DATASET]))
        b2t_pubmed_df = pd.DataFrame(records)
        b2t_pubmed_df["dataset"] = "pubmed"
        b2t_frames.append(b2t_pubmed_df)

    if RUN_NEUROVAULT:
        records = []
        for d in tqdm(neurovault_eval, desc="NeuroVault B2T"):
            records.extend(run_b2t(str(d["doi"]), d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_GENERAL, LONG_PROMPT))
        b2t_nv_df = pd.DataFrame(records)
        b2t_nv_df["dataset"] = "neurovault"
        b2t_frames.append(b2t_nv_df)

    b2t_all = pd.concat(b2t_frames, ignore_index=True)
    b2t_all.to_csv(OUTPUT_DIR / "brain_to_text_metrics_v2.csv", index=False)
    b2t_all.head()
else:
    print("Skipping generated text evaluation because RUN_GENERATED_TEXT = False.")
    b2t_all = pd.DataFrame()


### Generated Text Summary Tables

These tables summarize optional LLM-generation metrics. `bert_f1` measures contextual text overlap with the reference, `sem_sim` measures MiniLM sentence similarity, `nvlm_sim` measures similarity in NeuroVLM's shared latent space, and `generated_text_normalized_k_recall_curve_auc` measures whether generated texts retrieve their source brain maps early across the full normalized `k / n` retrieval curve within each dataset/mode group.


In [ ]:
from neurovlm.brain_to_text_metrics import generated_text_metric_summary

if RUN_GENERATED_TEXT and len(b2t_all):
    summary, generated_text_recall_auc_df, generated_text_recall_curve_df, label_summary = generated_text_metric_summary(
        nvlm=nvlm,
        b2t_all=b2t_all,
        networks_data=networks_data,
        pubmed_eval=pubmed_eval,
        neurovault_eval=neurovault_eval,
        output_dir=OUTPUT_DIR,
    )
    display(summary)
    display(generated_text_recall_auc_df.round(3))
    if len(label_summary):
        display(label_summary)
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


### Generated Text Normalized Recall Curves

This plot shows the full generated-text retrieval curve. Each generated output is embedded back into the NeuroVLM text space and ranked against the brain maps in the same dataset/mode group; the AUC summarizes Recall@k over normalized `k / n`, so groups with different sizes are comparable.


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all) and len(generated_text_recall_curve_df):
    fig, ax = plt.subplots(figsize=(8, 5))
    for (dataset, mode), curve_df in generated_text_recall_curve_df.groupby(["dataset", "mode"]):
        auc_row = generated_text_recall_auc_df[
            (generated_text_recall_auc_df["dataset"] == dataset) &
            (generated_text_recall_auc_df["mode"] == mode)
        ]
        auc = float(auc_row["generated_text_normalized_k_recall_curve_auc"].iloc[0]) if len(auc_row) else np.nan
        ax.plot(
            curve_df["normalized_k"],
            curve_df["recall_at_normalized_k"],
            label=f"{dataset} {mode} (AUC={auc:.3f})",
        )
    ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random chance")
    ax.set_xlabel("Normalized k (k / n brain maps)")
    ax.set_ylabel("Generated-text recall@k")
    ax.set_title("Generated Text Normalized Brain Retrieval Curves")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_generated_text_normalized_recall_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text normalized recall curve because RUN_GENERATED_TEXT = False or no curve rows are available.")


### Generated Text Metric Distributions

These plots show the distribution of generated-text scores by dataset and output mode, making it easier to see whether an average is driven by consistent behavior or a few strong/weak examples.


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    plot_df = b2t_all.copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metric_specs = [("nvlm_sim", "NeuroVLM latent similarity"), ("bert_f1", "BERTScore F1"), ("sem_sim", "Sentence semantic similarity")]
    for ax, (metric, title) in zip(axes, metric_specs):
        groups = [g[metric].dropna().values for _, g in plot_df.groupby("dataset")]
        labels = [k for k, _ in plot_df.groupby("dataset")]
        ax.boxplot(groups, labels=labels, showmeans=True)
        ax.set_title(title)
        ax.set_ylim(min(-0.05, np.nanmin(plot_df[metric]) - 0.05), min(1.05, np.nanmax(plot_df[metric]) + 0.1))
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_metric_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


### NeuroVLM Similarity Scale Check

This visualization compares matched generated-text/brain similarities with random pairings. It gives `nvlm_sim` an empirical baseline, which is more useful than reading cosine similarity as if only values near 1.0 were meaningful.


In [ ]:
from neurovlm.brain_to_text_metrics import generated_text_pair_baseline

if RUN_GENERATED_TEXT and len(b2t_all):
    baseline_df = generated_text_pair_baseline(
        nvlm=nvlm,
        b2t_all=b2t_all,
        networks_data=networks_data,
        pubmed_eval=pubmed_eval,
        neurovault_eval=neurovault_eval,
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    if len(baseline_df):
        matched = baseline_df[baseline_df["pair"] == "matched"]["score"]
        random_pairs = baseline_df[baseline_df["pair"] == "random/off-diagonal"]["score"]
        bins = np.linspace(min(baseline_df["score"].min(), 0), baseline_df["score"].max(), 24)
        ax.hist(random_pairs, bins=bins, alpha=0.55, label="random/off-diagonal pairs", color="lightgray", edgecolor="white")
        ax.hist(matched, bins=bins, alpha=0.75, label="matched generated text", color="steelblue", edgecolor="white")
        for x, lab, color in [(matched.median(), "matched median", "steelblue"), (random_pairs.median(), "random median", "gray")]:
            ax.axvline(x, color=color, linestyle="--", linewidth=1)
            ax.text(x, ax.get_ylim()[1] * 0.92, f"{lab}\n{x:.3f}", ha="center", va="top", fontsize=8)
    else:
        vals = b2t_all["nvlm_sim"].dropna()
        ax.hist(vals, bins=min(20, max(5, len(vals) // 2)), alpha=0.75, color="steelblue", edgecolor="white")
        ax.text(0.5, 0.9, "Need at least two samples per group for random-pair baseline", transform=ax.transAxes, ha="center")
    ax.set_title("How to read nvlm_sim: matched text versus random pairs")
    ax.set_xlabel("NeuroVLM latent cosine similarity")
    ax.set_ylabel("Pairs")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_nvlm_sim_scale.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


### Network Label Accuracy Visualization

For generated Network outputs, this plot and confusion matrix show whether the generated text names the correct canonical network label, using alias matching first and semantic matching as a fallback.


In [ ]:
if RUN_GENERATED_TEXT and len(b2t_all):
    if "network_label_correct" in b2t_all.columns:
        net = b2t_all[b2t_all["dataset"] == "networks"].copy()
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        acc = net.groupby("mode")["network_label_correct"].mean()
        axes[0].bar(acc.index, acc.values, color=["#4c78a8", "#72b7b2"])
        axes[0].set_ylim(0, 1)
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Network label accuracy")
        axes[0].grid(axis="y", alpha=0.25)

        cm = pd.crosstab(net["true_network_key"], net["pred_network_key"], normalize="index")
        im = axes[1].imshow(cm.values, vmin=0, vmax=1, cmap="Blues")
        axes[1].set_xticks(range(len(cm.columns)), cm.columns, rotation=45, ha="right")
        axes[1].set_yticks(range(len(cm.index)), cm.index)
        axes[1].set_title("Predicted network label by true label")
        fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        fig.tight_layout()
        plt.savefig(OUTPUT_DIR / "b2t_network_label_accuracy.png", dpi=150, bbox_inches="tight")
        plt.show()

        display(net[["name", "mode", "true_network_key", "pred_network_key", "network_label_correct", "label_match_method", "generated"]])
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")
